In [29]:
from pyspark.sql.types import StructType, IntegerType, StringType, DoubleType, StructField, TimestampType
import os
from pathlib import Path
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

In [30]:
# Ensure PySpark can find Homebrew Java inside this notebook kernel.
os.environ["JAVA_HOME"] = "/opt/homebrew/opt/openjdk/libexec/openjdk.jdk/Contents/Home"
os.environ["PATH"] = f"{os.environ['JAVA_HOME']}/bin:{os.environ['PATH']}"
repo_root = Path.cwd().parent
output_loc = str(repo_root/"yellowline"/"data"/"output")

spark = (
    SparkSession.builder
    .appName("notebook-spark-session")
    .master("local[*]")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.extraJavaOptions", "-Djava.security.manager=allow")
    .config("spark.executor.extraJavaOptions", "-Djava.security.manager=allow")
    .getOrCreate()
)


In [31]:

SILVER_SCHEMA = StructType(
    [
        StructField("VendorID",              IntegerType(),   True),
        StructField("tpep_pickup_datetime",  TimestampType(), True),
        StructField("tpep_dropoff_datetime", TimestampType(), True),
        StructField("passenger_count",       IntegerType(),   True),
        StructField("trip_distance",         DoubleType(),    True),
        StructField("PULocationID",          IntegerType(),   True),
        StructField("DOLocationID",          IntegerType(),   True),
        StructField("pickup_borough",        StringType(),    True),
        StructField("pickup_zone",           StringType(),    True),
        StructField("dropoff_borough",       StringType(),    True),
        StructField("dropoff_zone",          StringType(),    True),
        StructField("fare_amount",           DoubleType(),    True),
        StructField("tip_amount",            DoubleType(),    True),
        StructField("total_amount",          DoubleType(),    True),
        StructField("revenue_per_mile",      DoubleType(),    True),
    ]
)


In [36]:
bronze_path = f'{output_loc}/bronze'
bronze_df = spark.read.parquet(bronze_path)
bronze_df.show(5)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-------------+-----+-------+----------+------------+---------------------+------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|  fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-------------+-----+-------+----------+------------+---------------------+------------+
|       2| 2015-01-07 01:39:46|  2015-01-07 01:45:54|              1|         1.46|      NULL|40.760890960693359|           1|        NULL|        NULL|  40.76512146|  2.0|    7.0|       0.5|         0.5|                  0.0|         0.0|
|       2| 2015-01-07 01:39:49|  2015-01

In [34]:
silver_path = f'{output_loc}/silver'
silver_df = spark.read.format("parquet").schema(SILVER_SCHEMA).load(silver_path)
silver_df.show(5, truncate=False)
silver_df.createOrReplaceTempView("silver_temp")


+--------+--------------------+---------------------+---------------+-------------+------------+------------+--------------+-----------+---------------+------------+-----------+----------+------------+----------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|PULocationID|DOLocationID|pickup_borough|pickup_zone|dropoff_borough|dropoff_zone|fare_amount|tip_amount|total_amount|revenue_per_mile|
+--------+--------------------+---------------------+---------------+-------------+------------+------------+--------------+-----------+---------------+------------+-----------+----------+------------+----------------+
+--------+--------------------+---------------------+---------------+-------------+------------+------------+--------------+-----------+---------------+------------+-----------+----------+------------+----------------+



In [22]:
res = spark.sql("SELECT * FROM silver_temp WHERE PULocationID is not NULL AND DOLocationID is not NULL")

In [44]:
TRIP_SCHEMA = StructType(
    [
        StructField("VendorID", IntegerType(), True),
        StructField(
            "tpep_pickup_datetime", StringType(), True
        ),  # cast to timestamp in Silver
        StructField("tpep_dropoff_datetime", StringType(), True),
        StructField("passenger_count", IntegerType(), True),
        StructField("trip_distance", DoubleType(), True),
        StructField("RatecodeID", IntegerType(), True),
        StructField("store_and_fwd_flag", StringType(), True),
        StructField("PULocationID", IntegerType(), True),
        StructField("DOLocationID", DoubleType(), True),
        StructField("payment_type", IntegerType(), True),
        StructField("fare_amount", DoubleType(), True),
        StructField("extra", DoubleType(), True),
        StructField("mta_tax", DoubleType(), True),
        StructField("tip_amount", DoubleType(), True),
        StructField("tolls_amount", DoubleType(), True),
        StructField("improvement_surcharge", DoubleType(), True),
        StructField("total_amount", DoubleType(), True),
    ]
)

raw_df = spark.read.format('json').schema(TRIP_SCHEMA).load(str(repo_root)+ "/yellowline/data/input")
raw_df.show(5)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-------------+-----+-------+----------+------------+---------------------+------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|  fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-------------+-----+-------+----------+------------+---------------------+------------+
|       2| 2015-01-22 19:48:49|  2015-01-22 20:02:51|              2|         2.21|      NULL|40.750640869140625|           1|       208.0|        NULL|40.7367935181|  1.0|   10.5|       1.0|         0.5|                 2.46|         0.0|
|       2| 2015-01-22 19:48:49|  2015-01

In [43]:
raw_df.createOrReplaceTempView("raw_temp_view")
nonNullPUandDO = spark.sql("select * from raw_temp_view where PULocationID is not NULL and DOLocationID is not null")
nonNullPUandDO.show(5)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+



In [ ]:
import shutil

dirs_to_clear = [
    repo_root / "yellowline" / "data" / "input",
    repo_root / "yellowline" / "data" / "checkpoints" / "bronze",
    repo_root / "yellowline" / "data" / "checkpoints" / "silver",
    repo_root / "yellowline" / "data" / "output" / "bronze",
    repo_root / "yellowline" / "data" / "output" / "silver",
]

for d in dirs_to_clear:
    if d.exists():
        shutil.rmtree(d)
        d.mkdir(parents=True, exist_ok=True)
        print(f"Cleared: {d}")
    else:
        print(f"Does not exist, skipping: {d}")